# 03 — Train a 5-second JSBSim skill Transformer

This notebook consumes the canonical Parquet trajectories and label map produced by
`02_generate_jsbsim_skill_dataset.ipynb`. Complete flights are assigned to an
approximately **0.6:0.2:0.2 train:test:validation split**, stratified by the complete
set of tactical skills demonstrated in each flight.

Only `MODEL_FEATURE_COLUMNS` are model inputs. Commanded skills, native actions, and
other privileged columns are targets or audit metadata only. Transition-spanning
windows are excluded by default; set `BVR_KEEP_MIXED_WINDOWS=1` to label them by their
final sample.

The Transformer is trained on the training split. In accordance with this experiment's
selection protocol, test loss controls checkpointing and early stopping. The validation
split remains untouched until the selected checkpoint receives its final evaluation.
MLflow records configuration, per-epoch metrics, the checkpoint, and the complete
reproducibility bundle.

## 1. Imports, reproducibility, and MLflow configuration

The dataset path exactly matches notebook 02's default output. Environment variables
allow short, reproducible experiments without editing cells. CPU is the safe default;
set `BVR_TRAIN_DEVICE=cuda` only after verifying the local CUDA installation.

In [2]:
from __future__ import annotations

import gc
import hashlib
import json
import logging
import math
import os
import random
import shutil
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

# Git metadata is optional. Keep GitPython and MLflow quiet on machines where Git
# is not installed; runs still retain all explicitly logged parameters and metrics.
if shutil.which("git") is None:
    os.environ.setdefault("GIT_PYTHON_REFRESH", "quiet")
    logging.getLogger("mlflow.utils.git_utils").setLevel(logging.ERROR)

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.dataset as pads
import pyarrow.parquet as pq
import torch
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch import nn
from torch.utils.checkpoint import checkpoint as torch_checkpoint
from torch.utils.data import DataLoader

from bvr_behavior_prediction.data.observable_columns import MODEL_FEATURE_COLUMNS
from bvr_behavior_prediction.data.privileged_columns import PRIVILEGED_COLUMNS

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "pyproject.toml").exists():
    REPO_ROOT = REPO_ROOT.parent

DATASET_DIR = Path(os.getenv(
    "BVR_TRAIN_DATASET",
    "C:/Users/theon/Trajectory_Classification/datasets/bvr_f16_1v1_jsbsim_skills_v001",
))
OUTPUT_DIR = Path(os.getenv(
    "BVR_CLASSIFIER_OUTPUT", REPO_ROOT / "artifacts/models/jsbsim_skill_transformer_v001"
))
MLFLOW_DB_PATH = (REPO_ROOT / "artifacts/mlflow.db").resolve()
MLFLOW_DB_PATH.parent.mkdir(parents=True, exist_ok=True)
MLFLOW_TRACKING_URI = os.getenv(
    "BVR_MLFLOW_TRACKING_URI", f"sqlite:///{MLFLOW_DB_PATH.as_posix()}"
)
MLFLOW_EXPERIMENT = os.getenv("BVR_MLFLOW_EXPERIMENT", "jsbsim-skill-transformer")
WINDOW_S = 5.0
STRIDE_S = float(os.getenv("BVR_WINDOW_STRIDE_S", "1.0"))
KEEP_MIXED_WINDOWS = os.getenv("BVR_KEEP_MIXED_WINDOWS", "0") == "1"
BATCH_SIZE = int(os.getenv("BVR_BATCH_SIZE", "256"))
MICRO_BATCH_SIZE = int(os.getenv("BVR_MICRO_BATCH_SIZE", str(BATCH_SIZE)))
# Evaluation has no activation/gradient storage, so spend otherwise idle GPU memory on a
# larger physical batch. This reduces test/validation launch and input overhead without
# changing optimization or checkpoint selection.
EVAL_BATCH_SIZE = int(os.getenv("BVR_EVAL_BATCH_SIZE", str(max(BATCH_SIZE, 1024))))
ACCUMULATION_STEPS = math.ceil(BATCH_SIZE / MICRO_BATCH_SIZE)
GRADIENT_CHECKPOINTING = os.getenv("BVR_GRADIENT_CHECKPOINTING", "0") == "1"
PREPROCESS_CHUNK_WINDOWS = int(os.getenv("BVR_PREPROCESS_CHUNK_WINDOWS", "8192"))
PREPROCESS_WORKERS = int(os.getenv(
    "BVR_PREPROCESS_WORKERS", str(min(12, os.cpu_count() or 1))
))
WINDOW_BUILD_WORKERS = int(os.getenv(
    "BVR_WINDOW_BUILD_WORKERS", str(min(16, os.cpu_count() or 1))
))
ARROW_BATCH_ROWS = int(os.getenv("BVR_ARROW_BATCH_ROWS", "262144"))
FAST_CANONICAL_SCAN = os.getenv("BVR_FAST_CANONICAL_SCAN", "1") == "1"
METADATA_SCAN_WORKERS = int(os.getenv(
    "BVR_METADATA_SCAN_WORKERS", str(min(12, os.cpu_count() or 1))
))
METADATA_BATCH_ROWS = int(os.getenv("BVR_METADATA_BATCH_ROWS", "4096"))
KEEP_WINDOW_CACHE = os.getenv("BVR_KEEP_WINDOW_CACHE", "1") == "1"
DATALOADER_WORKERS = int(os.getenv(
    "BVR_DATALOADER_WORKERS", str(min(8, os.cpu_count() or 1))
))
DATALOADER_PREFETCH = int(os.getenv("BVR_DATALOADER_PREFETCH", "4"))
PRELOAD_FEATURES = os.getenv("BVR_PRELOAD_FEATURES", "1") == "1"
COMPILE_MODEL = os.getenv("BVR_COMPILE_MODEL", "1") == "1"
CUDA_PREFETCH = os.getenv("BVR_CUDA_PREFETCH", "1") == "1"
# When the compact trajectory arrays fit in GPU memory, gather overlapping windows on
# device instead of rebuilding and transferring them in CPU DataLoader workers each epoch.
CUDA_RESIDENT_DATASET = os.getenv("BVR_CUDA_RESIDENT_DATASET", "1") == "1"
USE_AMP = os.getenv("BVR_MIXED_PRECISION", "1") == "1"
TORCH_THREADS = int(os.getenv("BVR_TORCH_THREADS", str(min(12, os.cpu_count() or 1))))
REQUESTED_DEVICE = os.getenv("BVR_TRAIN_DEVICE", "cpu").strip().lower()
EPOCHS = int(os.getenv("BVR_TRAIN_EPOCHS", "30"))
PATIENCE = int(os.getenv("BVR_EARLY_STOPPING_PATIENCE", "6"))
LEARNING_RATE = float(os.getenv("BVR_LEARNING_RATE", "0.001"))
D_MODEL = int(os.getenv("BVR_TRANSFORMER_D_MODEL", "128"))
NHEAD = int(os.getenv("BVR_TRANSFORMER_HEADS", "8"))
NUM_LAYERS = int(os.getenv("BVR_TRANSFORMER_LAYERS", "3"))
DROPOUT = float(os.getenv("BVR_TRANSFORMER_DROPOUT", "0.2"))
SEED = int(os.getenv("BVR_TRAIN_SEED", "20260911"))
SPLIT_FRACTIONS = {"train": 0.60, "test": 0.20, "validation": 0.20}

if min(
    BATCH_SIZE, MICRO_BATCH_SIZE, EVAL_BATCH_SIZE, PREPROCESS_CHUNK_WINDOWS,
    ARROW_BATCH_ROWS, WINDOW_BUILD_WORKERS, DATALOADER_PREFETCH,
) < 1 or DATALOADER_WORKERS < 0:
    raise ValueError("Batch size must be positive and data-loader workers cannot be negative")
if BATCH_SIZE % MICRO_BATCH_SIZE or MICRO_BATCH_SIZE > BATCH_SIZE:
    raise ValueError("BVR_MICRO_BATCH_SIZE must divide BVR_BATCH_SIZE and cannot exceed it")
if METADATA_SCAN_WORKERS < 1:
    raise ValueError("BVR_METADATA_SCAN_WORKERS must be at least 1")
if METADATA_BATCH_ROWS < 1:
    raise ValueError("BVR_METADATA_BATCH_ROWS must be at least 1")
if PREPROCESS_WORKERS < 1:
    raise ValueError("BVR_PREPROCESS_WORKERS must be at least 1")
if TORCH_THREADS < 1:
    raise ValueError("BVR_TORCH_THREADS must be at least 1")
if D_MODEL % NHEAD:
    raise ValueError("BVR_TRANSFORMER_D_MODEL must be divisible by BVR_TRANSFORMER_HEADS")
if REQUESTED_DEVICE.startswith("cuda") and not torch.cuda.is_available():
    raise RuntimeError(
        f"BVR_TRAIN_DEVICE={REQUESTED_DEVICE!r} requires CUDA, but this PyTorch "
        "installation cannot access it. Use BVR_TRAIN_DEVICE=cpu or install a "
        "CUDA-compatible PyTorch build."
    )

torch.set_num_threads(TORCH_THREADS)
# Allow TensorFloat-32 tensor cores for float32 CUDA matmuls; AMP remains faster where supported.
torch.set_float32_matmul_precision("high")
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if REQUESTED_DEVICE.startswith("cuda"):
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device(REQUESTED_DEVICE)
AMP_ENABLED = USE_AMP and DEVICE.type == "cuda"
if DEVICE.type == "cuda":
    # TF32 speeds supported matrix multiplications without increasing memory use.
    torch.set_float32_matmul_precision("high")
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT)
print({"dataset": str(DATASET_DIR), "device": str(DEVICE), "seed": SEED,
       "mlflow_tracking_uri": mlflow.get_tracking_uri()})

{'dataset': 'C:\\Users\\theon\\Trajectory_Classification\\datasets\\bvr_f16_1v1_jsbsim_skills_v001', 'device': 'cpu', 'seed': 20260911, 'mlflow_tracking_uri': 'sqlite:///C:/Users/theon/Trajectory_Classification/artifacts/mlflow.db'}


In [3]:
print(f'METADATA_SCAN_WORKERS  : {METADATA_SCAN_WORKERS}  \n'  +
f'DATALOADER_WORKERS  : {DATALOADER_WORKERS}  \n' +
f'PREPROCESS_WORKERS   : {PREPROCESS_WORKERS}  \n'  +
f'TORCH_THREADS   : {TORCH_THREADS}  \n'
f'WINDOW_BUILD_WORKERS   : {WINDOW_BUILD_WORKERS}  \n')

METADATA_SCAN_WORKERS  : 12  
DATALOADER_WORKERS  : 0  
PREPROCESS_WORKERS   : 12  
TORCH_THREADS   : 12  
WINDOW_BUILD_WORKERS   : 16  



## 2. Load dataset metadata without materialising trajectories

The preferred cold path reads the compact `episodes.parquet` index plus Parquet footer row
counts. For datasets produced by notebook 02, each episode summary already contains its skill
schedule and duration, so decompressing the three metadata columns from every trajectory row is
unnecessary. Only the first two `time_s` values are decoded to establish cadence, trajectory footers are
read concurrently, and the episode index is consumed in bounded record batches. This path is
usually limited by footer reads and keeps peak memory proportional to the required episode
summaries plus one small batch, not the trajectory dataset.

If the compact index is absent or incompatible, the fallback scan is parallelized by Parquet
shard. Each worker projects only the three metadata columns and streams bounded record batches,
so the additional peak memory is roughly one small Arrow batch plus one episode per worker—not a
copy of the dataset. Set `BVR_METADATA_SCAN_WORKERS` to match storage throughput (the default is
at most four), and tune the compact-index batch with `BVR_METADATA_BATCH_ROWS` (default 4096).
Set `BVR_FORCE_METADATA_SCAN=1` to run the validating fallback deliberately.

The resulting episode summaries, sample count, and cadence are saved atomically in a
fingerprinted JSON cache. The fingerprint covers trajectory and episode-index file identity, the
label map, and relevant schema settings, so unchanged future runs skip all Parquet work. The
compact-index path is commonly an order of magnitude faster than a full cold scan, while warm
cache loads are commonly **10–100× or more** faster. The cell reports its actual path and elapsed
time so performance can be checked on the local storage.


In [ ]:
shards = sorted((DATASET_DIR / "trajectories").glob("*.parquet"))
label_map_path = DATASET_DIR / "label_map.json"
if not shards or not label_map_path.exists():
    raise FileNotFoundError(
        f"Expected trajectories/*.parquet and label_map.json under {DATASET_DIR}. "
        "Run notebooks/02_generate_jsbsim_skill_dataset.ipynb first or set BVR_TRAIN_DATASET."
    )

LABELS = json.loads(label_map_path.read_text())["tactical"]
label_to_index = {label: index for index, label in enumerate(LABELS)}
required_columns = ["episode_id", "time_s", "tactical_label", *MODEL_FEATURE_COLUMNS]
trajectory_dataset = pads.dataset(shards, format="parquet")
missing = set(required_columns).difference(trajectory_dataset.schema.names)
assert not missing, f"Missing required columns: {sorted(missing)}"
assert set(MODEL_FEATURE_COLUMNS).isdisjoint(PRIVILEGED_COLUMNS)

METADATA_CACHE_VERSION = 3
METADATA_CACHE_PATH = Path(os.getenv(
    "BVR_METADATA_CACHE", OUTPUT_DIR.parent / f".{OUTPUT_DIR.name}_metadata.json"
))
episodes_path = DATASET_DIR / "episodes.parquet"


def file_identity(path):
    if not path.exists():
        return None
    stat = path.stat()
    return {"name": path.name, "size": stat.st_size, "mtime_ns": stat.st_mtime_ns}


metadata_fingerprint = hashlib.sha256(json.dumps({
    "version": METADATA_CACHE_VERSION,
    "dataset_dir": str(DATASET_DIR.resolve()),
    "shards": [
        {"name": shard.name, "size": stat.st_size, "mtime_ns": stat.st_mtime_ns}
        for shard in shards for stat in [shard.stat()]
    ],
    "episodes": file_identity(episodes_path),
    "label_map_sha256": hashlib.sha256(label_map_path.read_bytes()).hexdigest(),
    "columns": ["episode_id", "time_s", "tactical_label"],
}, sort_keys=True).encode()).hexdigest()


def iter_dataset_episodes(dataset, columns):
    """Yield contiguous episodes while holding at most one scan batch plus one episode."""
    scanner = dataset.scanner(
        columns=columns, batch_size=ARROW_BATCH_ROWS, use_threads=True,
        batch_readahead=4, fragment_readahead=2,
    )
    pending_id, pending = None, {column: [] for column in columns}
    try:
        for batch in scanner.to_batches():
            arrays = {
                column: batch.column(column).to_numpy(zero_copy_only=False)
                for column in columns
            }
            ids = arrays["episode_id"]
            # Append the batch end to the actual ID changes. This avoids the two
            # temporary arrays created by np.r_ while preserving streaming order.
            boundaries = np.flatnonzero(ids[1:] != ids[:-1]) + 1
            left = 0
            for boundary_index in range(len(boundaries) + 1):
                right = boundaries[boundary_index] if boundary_index < len(boundaries) else len(ids)
                episode_id = ids[left]
                if pending_id is not None and episode_id != pending_id:
                    yield pending_id, {
                        column: np.concatenate(parts) if len(parts) > 1 else parts[0]
                        for column, parts in pending.items()
                    }
                    pending = {column: [] for column in columns}
                pending_id = episode_id
                for column in columns:
                    pending[column].append(arrays[column][left:right])
                left = right
        if pending_id is not None:
            yield pending_id, {
                column: np.concatenate(parts) if len(parts) > 1 else parts[0]
                for column, parts in pending.items()
            }
    except pa.ArrowKeyError as error:
        raise RuntimeError(
            "PyArrow's legacy extension registry is incompatible with this pandas session. "
            "Install the project dependencies (which require pyarrow>=14.0.1), restart the "
            "notebook kernel, and run all cells again."
        ) from error


def iter_episodes(columns):
    return iter_dataset_episodes(trajectory_dataset, columns)


def scan_metadata_shard(shard):
    """Summarize one shard; generated episodes never cross shard boundaries."""
    rows, shard_samples, shard_cadence = [], 0, None
    shard_dataset = pads.dataset([shard], format="parquet")
    for episode_id, episode in iter_dataset_episodes(
        shard_dataset, ["episode_id", "time_s", "tactical_label"]
    ):
        times = episode["time_s"]
        deltas = np.diff(times)
        episode_dt = float(np.median(deltas))
        assert episode_dt > 0 and np.allclose(
            deltas, episode_dt, atol=1e-6
        ), "Irregular sample cadence"
        if shard_cadence is not None:
            assert math.isclose(
                episode_dt, shard_cadence, abs_tol=1e-6
            ), "Inconsistent episode cadence"
        shard_cadence = episode_dt
        skills = tuple(sorted(set(episode["tactical_label"])))
        unknown = set(skills).difference(label_to_index)
        assert not unknown, f"Labels absent from label_map.json: {sorted(unknown)}"
        rows.append({"episode_id": episode_id, "skills": skills, "samples": len(times)})
        shard_samples += len(times)
    return rows, shard_samples, shard_cadence


def load_compact_metadata():
    """Use the canonical episode index and Parquet footers instead of a trajectory scan."""
    if os.getenv("BVR_FORCE_METADATA_SCAN", "0") == "1" or not episodes_path.exists():
        return None

    required_episode_columns = {"episode_id", "episode_duration_s", "skill_schedule_json"}
    episode_file = pq.ParquetFile(episodes_path)
    if not required_episode_columns.issubset(episode_file.schema_arrow.names):
        return None

    # A two-row batch recovers cadence without materialising an entire row group. Footer
    # counts provide the exact total and are independent I/O, so read them concurrently.
    first_file = pq.ParquetFile(shards[0])
    first_batch = next(first_file.iter_batches(batch_size=2, columns=["time_s"]), None)
    if first_batch is None or first_batch.num_rows < 2:
        return None
    first_time_values = first_batch.column("time_s").to_numpy(zero_copy_only=False)
    cadence = float(first_time_values[1] - first_time_values[0])
    if cadence <= 0:
        return None

    def parquet_row_count(shard):
        return pq.ParquetFile(shard).metadata.num_rows

    worker_count = min(METADATA_SCAN_WORKERS, len(shards))
    with ThreadPoolExecutor(max_workers=worker_count) as executor:
        sample_count = sum(executor.map(parquet_row_count, shards))

    # Keep only the summaries needed downstream. Iterating the compact index avoids the
    # temporary full Arrow table and full Python-record copy created by Table.to_pylist().
    episode_columns = sorted(required_episode_columns)
    rows = []
    for batch in episode_file.iter_batches(
        batch_size=METADATA_BATCH_ROWS, columns=episode_columns, use_threads=True
    ):
        columns = batch.to_pydict()
        records = zip(
            columns["episode_id"],
            columns["episode_duration_s"],
            columns["skill_schedule_json"],
        )
        for episode_id, episode_duration_s, schedule_json in records:
            schedule = json.loads(schedule_json)
            skills = tuple(sorted({schedule["primary"], schedule["secondary"]}))
            unknown = set(skills).difference(label_to_index)
            if unknown:
                raise ValueError(f"Labels absent from label_map.json: {sorted(unknown)}")
            samples = int(round(float(episode_duration_s) / cadence)) + 1
            rows.append({
                "episode_id": episode_id, "skills": skills, "samples": samples,
                "switch_time_s": float(schedule["switch_time_s"]),
                "primary_label": schedule["primary"], "secondary_label": schedule["secondary"],
            })

    if sum(row["samples"] for row in rows) != sample_count:
        # A non-canonical dataset may have early termination or variable cadence; retain
        # the complete validating scan as a safe compatibility path.
        return None
    if len({row["episode_id"] for row in rows}) != len(rows):
        raise ValueError("Episode IDs must be unique in episodes.parquet")
    return {"episode_rows": rows, "sample_count": sample_count, "cadence": cadence}


def load_metadata_cache():
    if not METADATA_CACHE_PATH.exists():
        return None
    cached = json.loads(METADATA_CACHE_PATH.read_text())
    if cached.get("fingerprint") != metadata_fingerprint:
        return None
    cached["episode_rows"] = [
        {**row, "skills": tuple(row["skills"])} for row in cached["episode_rows"]
    ]
    return cached


scan_started = time.perf_counter()
cached_metadata = load_metadata_cache()
if cached_metadata is None:
    compact_metadata = load_compact_metadata()
else:
    compact_metadata = None
if compact_metadata is not None:
    episode_rows = compact_metadata["episode_rows"]
    sample_count = compact_metadata["sample_count"]
    cadence = compact_metadata["cadence"]
    cached_metadata = {"fingerprint": metadata_fingerprint, **compact_metadata}
    metadata_source = "compact episode index + Parquet footers"
elif cached_metadata is None:
    worker_count = min(METADATA_SCAN_WORKERS, len(shards))
    episode_rows, sample_count, shard_cadences = [], 0, []
    with ThreadPoolExecutor(max_workers=worker_count) as executor:
        # Consume results as they arrive in shard order instead of retaining a
        # second, nested copy of every summary until the scan has completed.
        for rows, count, shard_cadence in executor.map(scan_metadata_shard, shards):
            episode_rows.extend(rows)
            sample_count += count
            if shard_cadence is not None:
                shard_cadences.append(shard_cadence)
    if not shard_cadences:
        raise ValueError("No trajectory samples were found")
    cadence = shard_cadences[0]
    assert all(
        math.isclose(value, cadence, abs_tol=1e-6) for value in shard_cadences
    ), "Inconsistent episode cadence"
    if len({row["episode_id"] for row in episode_rows}) != len(episode_rows):
        raise ValueError("Episode IDs must be unique across Parquet shards")
    cached_metadata = {
        "fingerprint": metadata_fingerprint,
        "episode_rows": episode_rows,
        "sample_count": sample_count,
        "cadence": cadence,
    }
    metadata_source = f"parallel scan ({worker_count} workers)"
else:
    episode_rows = cached_metadata["episode_rows"]
    sample_count = int(cached_metadata["sample_count"])
    cadence = float(cached_metadata["cadence"])
    metadata_source = "persistent cache"

if metadata_source != "persistent cache":
    METADATA_CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
    temporary_cache_path = METADATA_CACHE_PATH.with_suffix(METADATA_CACHE_PATH.suffix + ".tmp")
    temporary_cache_path.write_text(json.dumps(cached_metadata, separators=(",", ":")))
    os.replace(temporary_cache_path, METADATA_CACHE_PATH)

SAMPLE_DT_S = cadence
WINDOW_SAMPLES = int(round(WINDOW_S / SAMPLE_DT_S))
STRIDE_SAMPLES = max(1, int(round(STRIDE_S / SAMPLE_DT_S)))
assert WINDOW_SAMPLES >= 2
metadata_elapsed_s = time.perf_counter() - scan_started
print(
    f"Loaded {sample_count:,} samples from {len(episode_rows):,} episodes via "
    f"{metadata_source} in {metadata_elapsed_s:.3f}s"
)


In [ ]:
# Demonstrate that a later notebook run can reload the persisted outputs without rescanning.
reloaded_metadata = load_metadata_cache()
assert reloaded_metadata is not None
assert reloaded_metadata["sample_count"] == sample_count
assert math.isclose(reloaded_metadata["cadence"], SAMPLE_DT_S, abs_tol=1e-12)
assert reloaded_metadata["episode_rows"] == episode_rows
print(
    f"Reloaded cached metadata for {len(reloaded_metadata['episode_rows']):,} episodes from "
    f"{METADATA_CACHE_PATH.resolve()}"
)
del reloaded_metadata


## 3. Stratify complete episodes 60:20:20 by demonstrated skills

A flight can demonstrate more than one skill. Its stratum is therefore the sorted set
of all tactical labels appearing in that episode, rather than only its first or most
frequent label. A 60/40 stratified split is followed by an equal stratified division of
the remainder. This preserves joint skill combinations while keeping every flight—and
all overlapping windows from it—in exactly one split.

Every demonstrated-skill combination needs at least five episodes to be represented in
all three partitions. The production dataset from notebook 02 readily satisfies this;
a deliberately tiny smoke dataset fails with an actionable message rather than silently
falling back to an unstratified split.

In [ ]:
episode_skills = pd.DataFrame.from_records(episode_rows)
episode_skills["stratum"] = episode_skills["skills"].map("|".join)
stratum_counts = episode_skills["stratum"].value_counts()
if len(episode_skills) < 5 or (stratum_counts < 5).any():
    rare = stratum_counts[stratum_counts < 5].to_dict()
    raise ValueError(
        "Stratified 60:20:20 splitting requires at least five flights for every "
        f"demonstrated-skill combination; insufficient strata: {rare}. Generate more flights."
    )

train_episodes, selection_episodes = train_test_split(
    episode_skills,
    test_size=SPLIT_FRACTIONS["test"] + SPLIT_FRACTIONS["validation"],
    random_state=SEED,
    shuffle=True,
    stratify=episode_skills["stratum"],
)
test_episodes, validation_episodes = train_test_split(
    selection_episodes,
    test_size=0.5,
    random_state=SEED,
    shuffle=True,
    stratify=selection_episodes["stratum"],
)
split_tables = {
    "train": train_episodes,
    "test": test_episodes,
    "validation": validation_episodes,
}
episode_ids = episode_skills["episode_id"].to_numpy(copy=False)
episode_indices = {episode_id: index for index, episode_id in enumerate(episode_ids)}
split_ids = {name: table["episode_id"].tolist() for name, table in split_tables.items()}
all_split_ids = [set(ids) for ids in split_ids.values()]
assert all(all_split_ids) and not any(
    not all_split_ids[i].isdisjoint(all_split_ids[j])
    for i in range(len(all_split_ids)) for j in range(i + 1, len(all_split_ids))
)
assert set().union(*all_split_ids) == episode_indices.keys()

split_audit = pd.concat([
    table.assign(split=name).explode("skills")
    for name, table in split_tables.items()
], ignore_index=True)
split_summary = pd.crosstab(split_audit["skills"], split_audit["split"])
split_summary.loc["TOTAL EPISODES"] = {
    name: len(table) for name, table in split_tables.items()
}
print({name: round(len(ids) / len(episode_skills), 4) for name, ids in split_ids.items()})
display(split_summary)


## 4. Cache trajectories once and index five-second windows lazily

After splitting, a single streaming pass decodes features and labels, identifies admissible windows,
and retains the episode arrays until each trajectory sample is written once to a split-specific `.npy`
memory map. It does **not** materialize every overlapping window: with a five-second window and a
one-second stride, that old representation copied most samples about five times and made the section
after “Cache allocation” dominated by many GiB of avoidable disk writes.

Each window is now represented by one 64-bit start offset. `TrajectoryWindowDataset` takes a
zero-copy slice from the trajectory map when a DataLoader requests that window. This reduces cold-cache
feature writes by approximately `WINDOW_SAMPLES / STRIDE_SAMPLES`, shrinks the persistent cache by the
same factor when features dominate, and eliminates the large overlapping-window construction step.
Normalization is also performed once per trajectory sample rather than once per duplicated sample.
The trade-off is a small per-batch slicing cost and a compact start-offset array (eight bytes per window).
OS page caching benefits the overlapping accesses, while the substantially smaller cache makes better
use of available RAM.

A single Parquet pass processes independent shards concurrently and retains decoded episode columns in RAM
while it selects windows. After assigning disjoint feature and metadata ranges, parallel writers copy those
retained arrays into the memory maps without re-reading or re-decoding every Parquet shard. This intentionally
uses approximately one decoded dataset of additional peak memory to accelerate cache construction.
`BVR_WINDOW_BUILD_WORKERS` controls the copy parallelism. Completed arrays and counts are fingerprinted
and persisted atomically; unchanged runs reopen them and skip both scans.


### Fast canonical trajectory scan

For datasets produced by notebook 02, the compact episode index defines episode lengths and the two-label schedule. The cold window build therefore reads only numeric model features from trajectory shards, reconstructing episode boundaries, regular timestamps, and tactical labels from that index. In particular, it no longer reads the redundant `time_s` column because canonical samples start at zero and the validated dataset cadence defines every timestamp. This avoids decoding `episode_id` and `tactical_label` into per-row Python strings. The cell reports projected compressed bytes for the old and new projections, their I/O-bound speedup ceiling, and an estimate against the observed **24,556.06 s** baseline. It also reports the incremental I/O estimate from dropping `time_s`, so the benefit can be evaluated on the actual dataset rather than assumed. The optimized path also converts each numeric Arrow column once per shard rather than once per episode and derives mixed/target arrays directly from the numeric switch schedule. The runtime output reports the exact reduction in Arrow conversion calls (episodes per shard); this is a CPU-overhead ceiling, while the projection ratio remains the storage-bound estimate. CPU-bound runs may improve more because string conversion is also removed; storage-bound runs should approach the reported projection ratio. Increase `BVR_WINDOW_BUILD_WORKERS` only until storage throughput saturates (the default is at most 16), and set `BVR_FAST_CANONICAL_SCAN=0` to use the generic validating path. A warm run uses the persistent window cache and skips this scan altogether.


A synthetic CPU-only extraction benchmark (1,000 episodes × 301 rows × 22 numeric columns, repeated after warm-up) reduced conversion/slicing time from **0.191 s to 0.0108 s (17.7×)**. This isolates Python/Arrow extraction rather than claiming a 17.7× disk-scan improvement. By Amdahl's law, that corresponds to approximately **1.10×, 1.89×, or 6.76× end-to-end** if extraction represented 10%, 50%, or 90% of the measured scan stage. The canonical default of 1,000 flights per shard also produces **1,000× fewer Arrow conversion calls**; the notebook reports the actual factor for the loaded dataset. Treat the projection ratio as the I/O-bound estimate and these Amdahl cases as CPU-bound scenarios; compare the emitted stage timing on the target machine for the realized result.


In [ ]:
window_cell_started = time.perf_counter()
window_stage_seconds = {}
setup_started = time.perf_counter()

episode_to_split = {
    episode_id: split_name for split_name, ids in split_ids.items() for episode_id in ids
}
CACHE_DIR = Path(os.getenv(
    "BVR_WINDOW_CACHE", OUTPUT_DIR.parent / f".{OUTPUT_DIR.name}_window_cache"
))
WINDOW_CACHE_MANIFEST = CACHE_DIR / "manifest.json"
# Version 2 stores each trajectory sample once and represents windows as start offsets.
WINDOW_CACHE_VERSION = 2
window_cache_fingerprint = hashlib.sha256(json.dumps({
    "version": WINDOW_CACHE_VERSION,
    "metadata": metadata_fingerprint,
    "split_ids": split_ids,
    "features": list(MODEL_FEATURE_COLUMNS),
    "labels": LABELS,
    "window_samples": WINDOW_SAMPLES,
    "stride_samples": STRIDE_SAMPLES,
    "keep_mixed": KEEP_MIXED_WINDOWS,
}, sort_keys=True).encode()).hexdigest()
cache_suffixes = {
    "features": "features", "window_start": "window_start", "y": "y",
    "episode_index": "episode", "start_time_s": "start", "end_time_s": "end",
    "mixed_skill": "mixed",
}


def load_window_cache():
    """Open a complete, fingerprint-matched cache without loading arrays into RAM."""
    if not WINDOW_CACHE_MANIFEST.exists():
        return None
    try:
        manifest = json.loads(WINDOW_CACHE_MANIFEST.read_text())
        if manifest["fingerprint"] != window_cache_fingerprint:
            return None
        counts = {name: int(value) for name, value in manifest["window_counts"].items()}
        sample_counts = {name: int(value) for name, value in manifest["sample_counts"].items()}
        mixed = {name: int(value) for name, value in manifest["mixed_counts"].items()}
        arrays = {
            split_name: {
                key: np.load(CACHE_DIR / f"{split_name}_{suffix}.npy", mmap_mode="r+")
                for key, suffix in cache_suffixes.items()
            }
            for split_name in split_ids
        }
    except (OSError, ValueError, KeyError, json.JSONDecodeError):
        return None
    if any(
        len(arrays[name]["y"]) != counts[name]
        or len(arrays[name]["features"]) != sample_counts[name]
        for name in split_ids
    ):
        return None
    return arrays, counts, sample_counts, mixed, manifest


def window_starts(targets):
    starts = np.arange(0, len(targets) - WINDOW_SAMPLES + 1, STRIDE_SAMPLES)
    if not len(starts):
        return starts, np.empty(0, dtype=bool)
    changes = np.empty(len(targets), dtype=np.int32)
    changes[0] = 0
    np.cumsum(targets[1:] != targets[:-1], dtype=np.int32, out=changes[1:])
    mixed = changes[starts + WINDOW_SAMPLES - 1] != changes[starts]
    return starts, mixed


window_stage_seconds["setup/fingerprint"] = time.perf_counter() - setup_started
cache_lookup_started = time.perf_counter()
cached_windows = load_window_cache()
window_stage_seconds["cache lookup"] = time.perf_counter() - cache_lookup_started
if cached_windows is not None:
    raw, window_counts, split_sample_counts, mixed_counts, window_cache_manifest = cached_windows
    print(f"Reused {sum(window_counts.values()):,} indexed windows from {CACHE_DIR.resolve()}")
else:
    # Retain the feature columns during window selection. This trades available RAM for
    # eliminating the formerly dominant second Parquet decode/read pass.
    feature_columns = ["episode_id", "time_s", *MODEL_FEATURE_COLUMNS]
    selection_columns = [*feature_columns, "tactical_label"]

    # Canonical datasets store episodes in the same order in episodes.parquet and the
    # trajectory shards. Use footer row counts to map those compact summaries to shards.
    # This lets the hot scan project numeric columns only: repeated episode/label strings
    # no longer need to be decompressed or converted into millions of Python objects.
    def assign_canonical_episodes_to_shards():
        if not FAST_CANONICAL_SCAN or not all(
            {"switch_time_s", "primary_label", "secondary_label"}.issubset(row)
            for row in episode_rows
        ):
            return None
        assignments, episode_offset = {}, 0
        for shard in shards:
            shard_rows = pq.ParquetFile(shard).metadata.num_rows
            assigned, assigned_rows = [], 0
            while assigned_rows < shard_rows and episode_offset < len(episode_rows):
                row = episode_rows[episode_offset]
                assigned.append(row)
                assigned_rows += int(row["samples"])
                episode_offset += 1
            if assigned_rows != shard_rows:
                return None
            assignments[shard] = assigned
        return assignments if episode_offset == len(episode_rows) else None


    canonical_shard_episodes = assign_canonical_episodes_to_shards()
    scan_columns = list(MODEL_FEATURE_COLUMNS)

    def iter_canonical_shard_episodes(shard, summaries):
        """Decode each numeric column once, then expose zero-copy episode slices."""
        parquet_file = pq.ParquetFile(shard)
        table = parquet_file.read(columns=scan_columns, use_threads=False)
        arrays = {
            column: table[column].to_numpy(zero_copy_only=False) for column in scan_columns
        }
        left = 0
        for summary in summaries:
            right = left + int(summary["samples"])
            if right > table.num_rows:
                raise ValueError(f"Episode summaries exceed canonical shard {shard}")
            yield summary["episode_id"], {
                column: array[left:right] for column, array in arrays.items()
            }, summary
            left = right
        if left != table.num_rows:
            raise ValueError(f"Unconsumed rows in canonical shard {shard}")


    def canonical_window_selection(sample_count, summary, target_dtype):
        """Select windows from the validated cadence; canonical time_s is redundant."""
        starts = np.arange(0, sample_count - WINDOW_SAMPLES + 1, STRIDE_SAMPLES)
        if not len(starts):
            return starts, np.empty(0, dtype=bool), np.empty(0, dtype=target_dtype)
        # Canonical generation starts every episode at zero with a fixed cadence. The
        # epsilon preserves searchsorted(..., side="left") semantics at exact boundaries.
        switch_index = min(sample_count, max(0, int(math.ceil(
            summary["switch_time_s"] / SAMPLE_DT_S - 1e-12
        ))))
        ends = starts + WINDOW_SAMPLES - 1
        mixed = (starts < switch_index) & (ends >= switch_index)
        encoded_targets = np.where(
            ends < switch_index,
            label_to_index[summary["primary_label"]],
            label_to_index[summary["secondary_label"]],
        ).astype(target_dtype, copy=False)
        return starts, mixed, encoded_targets


    def parquet_projected_compressed_bytes(shard, columns):
        parquet_file = pq.ParquetFile(shard)
        indices = [parquet_file.schema_arrow.get_field_index(column) for column in columns]
        return sum(
            parquet_file.metadata.row_group(group).column(index).total_compressed_size
            for group in range(parquet_file.metadata.num_row_groups) for index in indices
        )


    old_projection_bytes = sum(
        parquet_projected_compressed_bytes(shard, selection_columns) for shard in shards
    )
    canonical_with_time_columns = ["time_s", *MODEL_FEATURE_COLUMNS]
    active_projection_columns = scan_columns if canonical_shard_episodes is not None else selection_columns
    active_projection_bytes = sum(
        parquet_projected_compressed_bytes(shard, active_projection_columns) for shard in shards
    )
    canonical_with_time_bytes = sum(
        parquet_projected_compressed_bytes(shard, canonical_with_time_columns) for shard in shards
    )
    projection_speedup_ceiling = old_projection_bytes / max(1, active_projection_bytes)
    time_elision_speedup = (
        canonical_with_time_bytes / max(1, active_projection_bytes)
        if canonical_shard_episodes is not None else 1.0
    )
    canonical_episode_count = sum(map(len, canonical_shard_episodes.values())) if canonical_shard_episodes else 0
    # Previously every episode converted every Arrow column separately. Bulk shard
    # conversion reduces those Python/C++ boundary crossings by episodes per shard.
    conversion_call_reduction = canonical_episode_count / len(shards) if canonical_episode_count else 1.0
    print(
        f"[windows] Scan path: {'numeric canonical' if canonical_shard_episodes else 'generic'}; "
        f"projected Parquet {old_projection_bytes / 1024**3:.2f} -> "
        f"{active_projection_bytes / 1024**3:.2f} GiB "
        f"({projection_speedup_ceiling:.2f}x I/O-bound ceiling, "
        f"{time_elision_speedup:.2f}x incremental from reconstructing time); "
        f"Arrow conversions {conversion_call_reduction:.1f}x fewer"
    )


    def prepare_window_shard(shard):
        counts = {name: 0 for name in split_ids}
        sample_counts = {name: 0 for name in split_ids}
        mixed = {name: 0 for name in split_ids}
        prepared_episodes = []
        target_dtype = np.min_scalar_type(max(0, len(LABELS) - 1))
        if canonical_shard_episodes is None:
            shard_dataset = pads.dataset([shard], format="parquet")
            episodes = iter_dataset_episodes(shard_dataset, selection_columns)
        else:
            episodes = iter_canonical_shard_episodes(shard, canonical_shard_episodes[shard])
        for record in episodes:
            if canonical_shard_episodes is None:
                episode_id, episode = record
                starts, is_mixed = window_starts(episode["tactical_label"])
                selected_targets = episode["tactical_label"][
                    starts.astype(np.intp, copy=False) + WINDOW_SAMPLES - 1
                ]
                encoded_targets = np.empty(len(selected_targets), dtype=target_dtype)
                matched_targets = np.zeros(len(selected_targets), dtype=bool)
                for label, label_index in label_to_index.items():
                    matches = selected_targets == label
                    encoded_targets[matches] = label_index
                    matched_targets |= matches
                if not matched_targets.all():
                    raise ValueError(f"Unknown target labels in episode {episode_id!r}")
            else:
                episode_id, episode, summary = record
                starts, is_mixed, encoded_targets = canonical_window_selection(
                    int(summary["samples"]), summary, target_dtype
                )
            split_name = episode_to_split[episode_id]
            mixed[split_name] += int(is_mixed.sum())
            if not KEEP_MIXED_WINDOWS:
                keep = ~is_mixed
                starts, is_mixed = starts[keep], is_mixed[keep]
                encoded_targets = encoded_targets[keep]
            starts = starts.astype(np.int32, copy=False)
            sample_count = (
                len(episode["time_s"]) if canonical_shard_episodes is None
                else int(summary["samples"])
            )
            counts[split_name] += len(starts)
            sample_counts[split_name] += sample_count
            prepared_episodes.append(
                (episode_id, split_name, sample_count, episode, starts,
                 is_mixed, encoded_targets)
            )
        return counts, sample_counts, mixed, prepared_episodes

    worker_count = min(WINDOW_BUILD_WORKERS, len(shards))
    count_started = time.perf_counter()
    with ThreadPoolExecutor(max_workers=worker_count) as executor:
        prepared_shards = list(executor.map(prepare_window_shard, shards))
    window_stage_seconds["Parquet scan/window selection"] = time.perf_counter() - count_started
    print(
        f"[windows] Parquet scan/window selection: "
        f"{window_stage_seconds['Parquet scan/window selection']:.2f}s ({worker_count} workers); "
        f"24556.06s baseline estimate: "
        f"{24556.06 / projection_speedup_ceiling:.2f}s at the I/O ceiling "
        f"(actual speedup also includes avoided Python string conversion)"
    )

    layout_started = time.perf_counter()
    window_counts = {name: 0 for name in split_ids}
    split_sample_counts = {name: 0 for name in split_ids}
    mixed_counts = {name: 0 for name in split_ids}
    episode_window_layout = {}
    episode_window_selection = {}
    for counts, sample_counts, mixed, prepared_episodes in prepared_shards:
        for name in split_ids:
            mixed_counts[name] += mixed[name]
        for episode_id, split_name, sample_count_for_episode, episode, starts, is_mixed, encoded_targets in prepared_episodes:
            if episode_id in episode_window_layout:
                raise ValueError(f"Episode {episode_id!r} occurs in more than one Parquet shard")
            window_count = len(starts)
            episode_window_layout[episode_id] = (
                split_name, window_counts[split_name], window_count,
                split_sample_counts[split_name], sample_count_for_episode,
            )
            episode_window_selection[episode_id] = (starts, is_mixed, encoded_targets)
            window_counts[split_name] += window_count
            split_sample_counts[split_name] += sample_count_for_episode
    window_stage_seconds["layout aggregation"] = time.perf_counter() - layout_started
    print(
        f"[windows] Layout aggregation: {window_stage_seconds['layout aggregation']:.2f}s; "
        f"{sum(window_counts.values()):,} windows index {sum(split_sample_counts.values()):,} samples"
    )
    if not all(window_counts.values()):
        raise ValueError("No windows were produced; check episode duration and filtering")

    allocation_started = time.perf_counter()
    shutil.rmtree(CACHE_DIR, ignore_errors=True)
    CACHE_DIR.mkdir(parents=True)
    raw = {}
    for split_name in split_ids:
        window_count = window_counts[split_name]
        source_count = split_sample_counts[split_name]
        specs = {
            "features": (np.float32, (source_count, len(MODEL_FEATURE_COLUMNS))),
            "window_start": (np.int64, (window_count,)),
            "y": (np.int64, (window_count,)),
            "episode_index": (np.int32, (window_count,)),
            "start_time_s": (np.float32, (window_count,)),
            "end_time_s": (np.float32, (window_count,)),
            "mixed_skill": (np.bool_, (window_count,)),
        }
        # Build in anonymous RAM, then serialize each completed array contiguously. Peak
        # memory is higher than direct memmap writes, but this avoids page faults and turns
        # 21 strided feature-column writes into one contiguous write per episode.
        raw[split_name] = {
            key: np.empty(shape, dtype=dtype) for key, (dtype, shape) in specs.items()
        }

    window_stage_seconds["cache allocation"] = time.perf_counter() - allocation_started
    allocated_gib = sum(array.nbytes for values in raw.values() for array in values.values()) / 1024**3
    old_feature_gib = (
        sum(window_counts.values()) * WINDOW_SAMPLES * len(MODEL_FEATURE_COLUMNS) * 4 / 1024**3
    )
    print(
        f"[windows] Cache allocation: {window_stage_seconds['cache allocation']:.2f}s "
        f"({allocated_gib:.2f} GiB; avoided {old_feature_gib:.2f} GiB overlapping feature cache)"
    )

    if episode_window_layout.keys() != episode_indices.keys():
        raise ValueError("Counted Parquet episodes do not match the split episode index")

    def write_prepared_shard(prepared_episodes):
        """Write retained source samples plus compact window offsets and metadata."""
        written = {name: 0 for name in split_ids}
        for (episode_id, _split_name, _sample_count, episode, _starts,
             _mixed, _encoded_targets) in prepared_episodes:
            split_name, window_left, expected_windows, sample_left, expected_samples = episode_window_layout[episode_id]
            starts, mixed, encoded_targets = episode_window_selection[episode_id]
            actual_samples = len(next(iter(episode.values())))
            if len(starts) != expected_windows or actual_samples != expected_samples:
                raise RuntimeError(f"Episode shape changed between passes for {episode_id!r}")
            sample_right = sample_left + expected_samples
            destination = raw[split_name]
            destination["features"][sample_left:sample_right] = np.column_stack(
                [episode[column] for column in MODEL_FEATURE_COLUMNS]
            )
            if not len(starts):
                continue
            window_right = window_left + len(starts)
            destination["window_start"][window_left:window_right] = sample_left + starts
            destination["y"][window_left:window_right] = encoded_targets
            destination["episode_index"][window_left:window_right] = episode_indices[episode_id]
            if canonical_shard_episodes is None:
                selected_start_times = episode["time_s"][starts]
                selected_end_times = episode["time_s"][starts + WINDOW_SAMPLES - 1]
            else:
                canonical_times = starts * SAMPLE_DT_S
                canonical_end_times = (starts + WINDOW_SAMPLES - 1) * SAMPLE_DT_S
                selected_start_times, selected_end_times = canonical_times, canonical_end_times
            destination["start_time_s"][window_left:window_right] = selected_start_times
            destination["end_time_s"][window_left:window_right] = selected_end_times
            destination["mixed_skill"][window_left:window_right] = mixed
            written[split_name] += len(starts)
        return written

    build_workers = min(WINDOW_BUILD_WORKERS, len(shards))
    written_counts = {name: 0 for name in split_ids}
    materialize_started = time.perf_counter()
    prepared_episode_shards = [prepared[3] for prepared in prepared_shards]
    with ThreadPoolExecutor(max_workers=build_workers) as executor:
        for shard_written in executor.map(write_prepared_shard, prepared_episode_shards):
            for name in split_ids:
                written_counts[name] += shard_written[name]
    window_stage_seconds["in-memory cache writes"] = time.perf_counter() - materialize_started
    print(
        f"[windows] In-memory cache writes: "
        f"{window_stage_seconds['in-memory cache writes']:.2f}s "
        f"({build_workers} workers, {sum(split_sample_counts.values()):,} unique samples)"
    )
    assert written_counts == window_counts
    del episode_window_selection, prepared_episode_shards, prepared_shards

    serialization_started = time.perf_counter()
    cache_arrays = [
        (CACHE_DIR / f"{split_name}_{cache_suffixes[key]}.npy", array)
        for split_name, values in raw.items() for key, array in values.items()
    ]

    def save_cache_array(item):
        path, array = item
        np.save(path, array, allow_pickle=False)

    # Separate files are independent. A small bounded pool overlaps file creation and
    # kernel copies while preserving large, contiguous writes for every .npy payload.
    serialization_workers = min(4, WINDOW_BUILD_WORKERS, len(cache_arrays))
    with ThreadPoolExecutor(max_workers=serialization_workers) as executor:
        tuple(executor.map(save_cache_array, cache_arrays))
    del raw, cache_arrays
    raw = {
        split_name: {
            key: np.load(CACHE_DIR / f"{split_name}_{suffix}.npy", mmap_mode="r+")
            for key, suffix in cache_suffixes.items()
        }
        for split_name in split_ids
    }
    window_stage_seconds["contiguous cache serialization"] = (
        time.perf_counter() - serialization_started
    )
    print(
        f"[windows] Contiguous cache serialization: "
        f"{window_stage_seconds['contiguous cache serialization']:.2f}s "
        f"({serialization_workers} workers)"
    )
    window_cache_manifest = {
        "fingerprint": window_cache_fingerprint,
        "window_counts": window_counts,
        "sample_counts": split_sample_counts,
        "mixed_counts": mixed_counts,
        "normalized": False,
        "representation": "trajectory_samples_with_window_offsets",
    }
    manifest_started = time.perf_counter()
    temporary_manifest = WINDOW_CACHE_MANIFEST.with_suffix(".tmp")
    temporary_manifest.write_text(json.dumps(window_cache_manifest, indent=2))
    os.replace(temporary_manifest, WINDOW_CACHE_MANIFEST)
    window_stage_seconds["manifest commit"] = time.perf_counter() - manifest_started

# Complete episode assignment already proves that overlapping windows cannot leak.
assert len(episode_to_split) == len(episode_indices) == len(episode_ids)
for split_name in split_ids:
    print(split_name, {"episodes": len(split_ids[split_name]),
                       "samples": split_sample_counts[split_name],
                       "windows": window_counts[split_name],
                       "excluded_mixed": mixed_counts[split_name]})
window_total_seconds = time.perf_counter() - window_cell_started
slowest_stage, slowest_seconds = max(window_stage_seconds.items(), key=lambda item: item[1])
print("[windows] Performance summary (wall time):")
for stage_name, elapsed_s in sorted(window_stage_seconds.items(), key=lambda item: item[1], reverse=True):
    print(f"  {stage_name:30s} {elapsed_s:9.2f}s  ({elapsed_s / window_total_seconds:6.1%})")
print(
    f"[windows] Slowest measured stage: {slowest_stage} ({slowest_seconds:.2f}s); "
    f"total cell {window_total_seconds:.2f}s"
)
gc.collect()


In [ ]:
# Demonstrate reloading every product without copying the disk-backed arrays into memory.
reloaded_window_cache = load_window_cache()
assert reloaded_window_cache is not None
(reloaded_raw, reloaded_window_counts, reloaded_sample_counts,
 reloaded_mixed_counts, reloaded_manifest) = reloaded_window_cache
assert reloaded_window_counts == window_counts
assert reloaded_sample_counts == split_sample_counts
assert reloaded_mixed_counts == mixed_counts
assert all(
    reloaded_raw[name][key].shape == raw[name][key].shape
    for name in split_ids for key in cache_suffixes
)
print(
    f"Reloaded {sum(reloaded_window_counts.values()):,} indexed windows over "
    f"{sum(reloaded_sample_counts.values()):,} samples from {CACHE_DIR.resolve()}"
)
raw = reloaded_raw
window_cache_manifest = reloaded_manifest
del (reloaded_window_cache, reloaded_raw, reloaded_window_counts,
     reloaded_sample_counts, reloaded_mixed_counts)


## 5. Incremental preprocessing and lazy window views

`StandardScaler.partial_fit` and in-place normalization operate on each unique trajectory sample,
not on duplicated overlapping windows. Independent split normalization remains bounded and parallel.
At training time, `TrajectoryWindowDataset` constructs only the requested window as a view of the
normalized feature memory map; DataLoader collation copies those views directly into the batch tensor.


In [ ]:
scaler = StandardScaler(copy=False)
if window_cache_manifest.get("normalized", False):
    normalization_mean = np.asarray(window_cache_manifest["normalization_mean"], dtype=np.float32)
    normalization_scale = np.asarray(window_cache_manifest["normalization_scale"], dtype=np.float32)
    scaler.mean_ = normalization_mean.astype(np.float64)
    scaler.scale_ = normalization_scale.astype(np.float64)
    scaler.var_ = scaler.scale_ ** 2
    scaler.n_features_in_ = len(normalization_mean)
    print("Reused normalized trajectory maps and cached scaler parameters")
else:
    train_features = raw["train"]["features"]
    for left in range(0, len(train_features), PREPROCESS_CHUNK_WINDOWS):
        scaler.partial_fit(train_features[left:left + PREPROCESS_CHUNK_WINDOWS])
    normalization_mean = scaler.mean_.astype(np.float32)
    normalization_scale = scaler.scale_.astype(np.float32)
    del train_features

    normalization_workers = min(PREPROCESS_WORKERS, len(raw), PREPROCESS_CHUNK_WINDOWS)
    normalization_chunk_rows = max(1, PREPROCESS_CHUNK_WINDOWS // normalization_workers)

    def normalize_split(values):
        features = values["features"]
        for left in range(0, len(features), normalization_chunk_rows):
            chunk = features[left:left + normalization_chunk_rows]
            chunk -= normalization_mean
            chunk /= normalization_scale
        features.flush()

    with ThreadPoolExecutor(max_workers=normalization_workers) as executor:
        tuple(executor.map(normalize_split, raw.values()))
    window_cache_manifest.update({
        "normalized": True,
        "normalization_mean": normalization_mean.tolist(),
        "normalization_scale": normalization_scale.tolist(),
    })
    temporary_manifest = WINDOW_CACHE_MANIFEST.with_suffix(".tmp")
    temporary_manifest.write_text(json.dumps(window_cache_manifest, indent=2))
    os.replace(temporary_manifest, WINDOW_CACHE_MANIFEST)


class TrajectoryWindowDataset(torch.utils.data.Dataset):
    """Resolve compact offsets to zero-copy trajectory slices on demand."""
    def __init__(self, values, window_samples, preload_features=PRELOAD_FEATURES):
        # Sequentially copy the normalized map once. This exchanges CPU RAM for avoiding
        # random mmap page faults during every shuffled epoch. The CUDA-resident path can
        # copy directly from the map and does not need a second host copy.
        self.features = (
            np.array(values["features"], copy=True, order="C")
            if preload_features else values["features"]
        )
        self.window_start = values["window_start"]
        self.targets = values["y"]
        self.window_samples = window_samples

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, index):
        start = int(self.window_start[index])
        # asarray avoids PyTorch's warning for the read/write memmap-backed view.
        features = torch.from_numpy(np.asarray(
            self.features[start:start + self.window_samples]
        ))
        return features, torch.tensor(int(self.targets[index]), dtype=torch.int64)


class CudaTrajectoryLoader:
    """Keep compact trajectories on GPU and assemble overlapping windows there."""
    device_resident = True

    def __init__(self, dataset, batch_size, shuffle, seed):
        self.dataset = dataset
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.generator = torch.Generator(device=DEVICE).manual_seed(seed)
        # Only unique trajectory samples are resident: windows remain compact offsets.
        self.features = torch.as_tensor(
            np.asarray(dataset.features), dtype=torch.float32, device=DEVICE
        )
        self.window_start = torch.as_tensor(
            np.asarray(dataset.window_start), dtype=torch.int64, device=DEVICE
        )
        self.targets = torch.as_tensor(
            np.asarray(dataset.targets), dtype=torch.int64, device=DEVICE
        )
        self.window_offsets = torch.arange(dataset.window_samples, device=DEVICE)

    def __len__(self):
        return math.ceil(len(self.dataset) / self.batch_size)

    def __iter__(self):
        if self.shuffle:
            order = torch.randperm(
                len(self.dataset), generator=self.generator, device=DEVICE
            )
        else:
            order = torch.arange(len(self.dataset), device=DEVICE)
        for left in range(0, len(self.dataset), self.batch_size):
            selection = order[left:left + self.batch_size]
            sample_indices = (
                self.window_start[selection, None] + self.window_offsets[None, :]
            )
            yield self.features[sample_indices], self.targets[selection]


loaders = {}
for split_name, values in raw.items():
    use_cuda_resident = CUDA_RESIDENT_DATASET and DEVICE.type == "cuda"
    dataset = TrajectoryWindowDataset(
        values, WINDOW_SAMPLES, preload_features=PRELOAD_FEATURES and not use_cuda_resident
    )
    batch_size = MICRO_BATCH_SIZE if split_name == "train" else EVAL_BATCH_SIZE
    if use_cuda_resident:
        loaders[split_name] = CudaTrajectoryLoader(
            dataset, batch_size, split_name == "train", SEED
        )
    else:
        generator = torch.Generator().manual_seed(SEED) if split_name == "train" else None
        loaders[split_name] = DataLoader(
            dataset, batch_size=batch_size, shuffle=split_name == "train",
            generator=generator, num_workers=DATALOADER_WORKERS,
            pin_memory=DEVICE.type == "cuda",
            persistent_workers=DATALOADER_WORKERS > 0,
            prefetch_factor=DATALOADER_PREFETCH if DATALOADER_WORKERS > 0 else None,
        )
print("Train windows:", len(loaders["train"].dataset),
      "shape:", (WINDOW_SAMPLES, len(MODEL_FEATURE_COLUMNS)),
      "effective train batch:", MICRO_BATCH_SIZE * ACCUMULATION_STEPS,
      "evaluation batch:", EVAL_BATCH_SIZE)


## 6. Throughput-first Transformer training tracked by MLflow

The defaults now deliberately spend the available memory: a 256-window physical batch, no
activation checkpointing, CPU workers with deeper prefetching, an in-RAM feature cache, AMP,
fused AdamW, TF32, optimized scaled-dot-product attention, `torch.compile`, and a dedicated CUDA
stream that overlaps the next pinned-memory transfer with current-batch compute. When the compact unique-sample
trajectory maps fit on the GPU, the default CUDA-resident loader goes further: it copies each
split once and gathers overlapping windows from their offsets on device, removing per-window
Python slicing, CPU collation, and repeated host-to-device traffic from every epoch. Evaluation uses a
separate 1,024-window default batch because inference does not retain backward activations; this
usually makes the per-epoch test pass **1.1–2.0× faster** when it was launch or input bound, with no
change to predictions or model selection. Set `BVR_EVAL_BATCH_SIZE` independently if it exhausts
GPU or CPU memory.
Checkpointing remains available for out-of-memory recovery, but it recomputes every encoder layer
in backward and is therefore not a throughput optimization. The first compiled epoch includes
one-time compilation; compare the reported steady-state epoch throughput rather than that epoch.

Expected **steady-state GPU acceleration is about 1.3–2.7×** versus the former defaults (batch 64,
checkpointing enabled, no workers, disk-backed random reads, eager execution). This is an
engineering estimate, not a benchmark result: removing recomputation has a compute-only ceiling
near 1.33×; compilation commonly contributes roughly 1.1–1.3× for a stable graph; and a 4× batch
plus prefetched RAM input can contribute roughly 1.0–1.5× depending on whether launch/input
latency was limiting. Asynchronous copy/compute overlap can add roughly 1.0–1.15× when transfers
are visible. A CUDA-resident trajectory cache is expected to add **1.05–1.4×** when input
assembly or PCIe transfer is material, and approximately 1.0× when attention is already
compute-bound; an unusually input-bound run can approach 2×. Do not multiply this range by
the other estimates because it replaces much of the work that prefetching and RAM preloading
already optimize.
Multiplying the non-resident-path midpoints suggests about 1.9× for the training pass, but the effects overlap and compute-bound hardware will
be near the low end. CPU training should expect a smaller, input- and shape-dependent gain. The notebook reports samples/s, peak GPU memory, and speedup relative to the
first steady-state epoch, so rerun both configurations for a hardware-specific estimate. For an
end-to-end epoch estimate, combine measured baseline phase times rather than multiplying all ranges:
`speedup = (baseline_train_s + baseline_test_s) / (baseline_train_s / train_speedup +
baseline_test_s / test_speedup)`. For example, if test evaluation was 20% of the baseline epoch, a
1.9× train-pass gain and 1.5× test-pass gain predict **about 1.81× end-to-end**.

Memory costs are explicit: preloading adds one copy of each split's feature map to CPU RAM; the
larger batch and disabled checkpointing retain more activations on GPU. The resident path adds one float32 copy of each split's unique samples plus compact offsets
to GPU memory (roughly `unique_samples × features × 4 bytes`, not one copy per window). Set
`BVR_CUDA_RESIDENT_DATASET=0` to return to the pinned-memory prefetch path. Set
`BVR_PRELOAD_FEATURES=0`, reduce `BVR_MICRO_BATCH_SIZE`, or set
`BVR_GRADIENT_CHECKPOINTING=1` if memory becomes limiting. Disable compilation with
`BVR_COMPILE_MODEL=0` for short CPU runs or unsupported backends; set
`BVR_CUDA_PREFETCH=0` to diagnose stream or driver compatibility.

After every epoch, **test loss** controls checkpointing and early stopping. MLflow receives
configuration, timing, throughput, and train/test metrics for every epoch.


In [ ]:
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_length, dropout):
        super().__init__()
        positions = torch.arange(max_length, dtype=torch.float32).unsqueeze(1)
        frequencies = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10_000.0) / d_model)
        )
        encoding = torch.zeros(max_length, d_model)
        encoding[:, 0::2] = torch.sin(positions * frequencies)
        encoding[:, 1::2] = torch.cos(positions * frequencies)
        self.register_buffer("encoding", encoding.unsqueeze(0), persistent=True)
        self.dropout = nn.Dropout(dropout)

    def forward(self, sequence):
        return self.dropout(sequence + self.encoding[:, :sequence.size(1)])


class SkillTransformer(nn.Module):
    def __init__(self, input_dim, class_count, window_samples, d_model=128,
                 nhead=8, layers=3, dropout=0.2):
        super().__init__()
        self.input_projection = nn.Linear(input_dim, d_model)
        self.positions = SinusoidalPositionalEncoding(d_model, window_samples, dropout)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=4 * d_model,
            dropout=dropout, activation="gelu", batch_first=True, norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(
            encoder_layer, num_layers=layers, norm=nn.LayerNorm(d_model),
            enable_nested_tensor=False,
        )
        self.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(d_model, class_count))

    def forward(self, sequence):
        encoded = self.positions(self.input_projection(sequence))
        if self.training and GRADIENT_CHECKPOINTING:
            for layer in self.encoder.layers:
                encoded = torch_checkpoint(layer, encoded, use_reentrant=False)
            if self.encoder.norm is not None:
                encoded = self.encoder.norm(encoded)
        else:
            encoded = self.encoder(encoded)
        return self.classifier(encoded.mean(dim=1))


model_config = {
    "input_dim": len(MODEL_FEATURE_COLUMNS), "class_count": len(LABELS),
    "window_samples": WINDOW_SAMPLES, "d_model": D_MODEL, "nhead": NHEAD,
    "layers": NUM_LAYERS, "dropout": DROPOUT,
}
model = SkillTransformer(**model_config).to(DEVICE)
compile_enabled = COMPILE_MODEL and DEVICE.type == "cuda"
execution_model = torch.compile(model, mode="max-autotune") if compile_enabled else model
counts = np.bincount(raw["train"]["y"], minlength=len(LABELS))
weights = counts.sum() / (len(LABELS) * np.maximum(counts, 1))
criterion = nn.CrossEntropyLoss(weight=torch.tensor(weights, dtype=torch.float32, device=DEVICE))
optimizer = torch.optim.AdamW(
    model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4, fused=DEVICE.type == "cuda"
)
grad_scaler = torch.amp.GradScaler("cuda", enabled=AMP_ENABLED)


class DevicePrefetcher:
    """Overlap pinned-memory H2D copies with compute on a dedicated CUDA stream."""
    def __init__(self, loader):
        self.iterator = iter(loader)
        self.stream = torch.cuda.Stream()
        self.next_batch = None
        self._preload()

    def _preload(self):
        try:
            features, target = next(self.iterator)
        except StopIteration:
            self.next_batch = None
            return
        with torch.cuda.stream(self.stream):
            self.next_batch = (
                features.to(DEVICE, non_blocking=True),
                target.to(DEVICE, non_blocking=True),
            )

    def __iter__(self):
        return self

    def __next__(self):
        if self.next_batch is None:
            raise StopIteration
        torch.cuda.current_stream().wait_stream(self.stream)
        features, target = self.next_batch
        # Tell the caching allocator that the tensors remain live on the compute stream.
        features.record_stream(torch.cuda.current_stream())
        target.record_stream(torch.cuda.current_stream())
        self._preload()
        return features, target


def device_batches(loader):
    if getattr(loader, "device_resident", False):
        return iter(loader)
    if CUDA_PREFETCH and DEVICE.type == "cuda":
        return DevicePrefetcher(loader)
    return (
        (features.to(DEVICE, non_blocking=True), target.to(DEVICE, non_blocking=True))
        for features, target in loader
    )


def run_epoch(loader, training=False):
    model.train(training)
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
    epoch_started = time.perf_counter()
    total = 0
    if DEVICE.type == "cuda":
        # Keep reductions device-side and synchronize only once per epoch rather
        # than calling .item() twice for every accelerator batch.
        totals = torch.zeros(2, dtype=torch.float64, device=DEVICE)
    else:
        total_loss = total_correct = 0
    if training:
        optimizer.zero_grad(set_to_none=True)
    for batch_index, (features, target) in enumerate(device_batches(loader)):
        with torch.set_grad_enabled(training), torch.autocast(
            device_type=DEVICE.type, dtype=torch.float16, enabled=AMP_ENABLED
        ):
            logits = execution_model(features)
            loss = criterion(logits, target)
        if training:
            grad_scaler.scale(loss / ACCUMULATION_STEPS).backward()
            update = (batch_index + 1) % ACCUMULATION_STEPS == 0 or batch_index + 1 == len(loader)
            if update:
                grad_scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                grad_scaler.step(optimizer)
                grad_scaler.update()
                optimizer.zero_grad(set_to_none=True)
        correct = (logits.detach().argmax(1) == target).sum()
        if DEVICE.type == "cuda":
            totals[0] += loss.detach() * len(target)
            totals[1] += correct
        else:
            total_loss += loss.detach().item() * len(target)
            total_correct += correct.item()
        total += len(target)
    if DEVICE.type == "cuda":
        total_loss, total_correct = totals.cpu().tolist()
        torch.cuda.synchronize()
        peak_gpu_memory_gib = torch.cuda.max_memory_allocated() / 1024**3
    else:
        peak_gpu_memory_gib = 0.0
    elapsed_s = time.perf_counter() - epoch_started
    return {
        "loss": total_loss / total, "accuracy": total_correct / total,
        "seconds": elapsed_s, "samples_per_second": total / elapsed_s,
        "peak_gpu_memory_gib": peak_gpu_memory_gib,
    }


OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = OUTPUT_DIR / "checkpoint.pt"
history, best_test_loss, stale_epochs = [], float("inf"), 0
mlflow_params = {
    **model_config, "architecture": "transformer_encoder", "batch_size": BATCH_SIZE,
    "evaluation_batch_size": EVAL_BATCH_SIZE,
    "micro_batch_size": MICRO_BATCH_SIZE, "accumulation_steps": ACCUMULATION_STEPS,
    "gradient_checkpointing": GRADIENT_CHECKPOINTING, "compile_model": compile_enabled,
    "cuda_prefetch": CUDA_PREFETCH and DEVICE.type == "cuda",
    "cuda_resident_dataset": CUDA_RESIDENT_DATASET and DEVICE.type == "cuda",
    "preload_features": PRELOAD_FEATURES, "dataloader_prefetch": DATALOADER_PREFETCH,
    "epochs": EPOCHS, "patience": PATIENCE, "learning_rate": LEARNING_RATE,
    "seed": SEED, "device": str(DEVICE), "mixed_precision": AMP_ENABLED,
    "dataloader_workers": DATALOADER_WORKERS, "window_s": WINDOW_S, "stride_s": STRIDE_S,
    **{f"split_{name}": fraction for name, fraction in SPLIT_FRACTIONS.items()},
}

with mlflow.start_run(run_name=f"transformer-seed-{SEED}") as active_run:
    run_id = active_run.info.run_id
    mlflow.log_params(mlflow_params)
    mlflow.set_tags({"dataset": DATASET_DIR.name, "checkpoint_selection_split": "test"})
    for epoch in range(1, EPOCHS + 1):
        train_metrics = run_epoch(loaders["train"], training=True)
        test_metrics = run_epoch(loaders["test"])
        row = {"epoch": epoch,
               **{f"train_{key}": value for key, value in train_metrics.items()},
               **{f"test_{key}": value for key, value in test_metrics.items()}}
        history.append(row)
        mlflow.log_metrics({key: value for key, value in row.items() if key != "epoch"}, step=epoch)
        print(f"{epoch:02d} train loss={train_metrics['loss']:.4f} "
              f"acc={train_metrics['accuracy']:.3f} test loss={test_metrics['loss']:.4f} "
              f"acc={test_metrics['accuracy']:.3f} train={train_metrics['seconds']:.1f}s "
              f"({train_metrics['samples_per_second']:.0f} samples/s, "
              f"peak GPU {train_metrics['peak_gpu_memory_gib']:.2f} GiB)")
        if test_metrics["loss"] < best_test_loss - 1e-4:
            best_test_loss = test_metrics["loss"]
            stale_epochs = 0
            torch.save({
                "epoch": epoch, "test_loss": best_test_loss,
                "model_state_dict": model.state_dict(), "model_config": model_config,
            }, CHECKPOINT_PATH)
        else:
            stale_epochs += 1
            if stale_epochs >= PATIENCE:
                print("Early stopping on test loss")
                break

    checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=True)
    model.load_state_dict(checkpoint["model_state_dict"])
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    mlflow.log_metric("best_test_loss", checkpoint["test_loss"])
    mlflow.log_metric("best_epoch", checkpoint["epoch"])
    mlflow.log_artifact(str(CHECKPOINT_PATH), artifact_path="checkpoints")

# Epoch 1 may contain torch.compile warm-up. Use epoch 2 as the steady-state reference
# when available, and expose relative throughput without claiming a cross-run benchmark.
steady_state_reference = history[1]["train_samples_per_second"] if len(history) > 1 else history[0]["train_samples_per_second"]
for row in history:
    row["train_speedup_vs_reference"] = row["train_samples_per_second"] / steady_state_reference
print(
    f"Steady-state reference: {steady_state_reference:,.0f} train samples/s; "
    "compare this value with a baseline run using BVR_BATCH_SIZE=64, "
    "BVR_GRADIENT_CHECKPOINTING=1, BVR_EVAL_BATCH_SIZE=64, "
    "BVR_DATALOADER_WORKERS=0, "
    "BVR_PRELOAD_FEATURES=0, BVR_COMPILE_MODEL=0, BVR_CUDA_PREFETCH=0, "
    "and BVR_CUDA_RESIDENT_DATASET=0."
)
history = pd.DataFrame(history)
history.plot(x="epoch", y=["train_loss", "test_loss"], grid=True,
             title="MLflow-tracked learning curves")
plt.show()
print({"mlflow_run_id": run_id, "selected_epoch": checkpoint["epoch"]})


## 7. Final evaluation on the untouched validation set

Validation is not used for scaling, optimization, checkpoint selection, or early
stopping. It provides the final class-wise report for the test-selected checkpoint.

In [ ]:
def predict(loader):
    model.eval()
    # Exact-size arrays avoid Python-object lists and their final full copies.
    actual = np.empty(len(loader.dataset), dtype=np.int64)
    predicted = np.empty_like(actual)
    offset = 0
    with torch.inference_mode():
        for features, target in device_batches(loader):
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=AMP_ENABLED):
                logits = execution_model(features)
            batch_size = len(target)
            actual[offset:offset + batch_size] = target.cpu().numpy()
            predicted[offset:offset + batch_size] = logits.argmax(1).cpu().numpy()
            offset += batch_size
    assert offset == len(actual)
    return actual, predicted


y_validation, y_pred = predict(loaders["validation"])
validation_accuracy = float((y_validation == y_pred).mean())
report = classification_report(
    y_validation, y_pred, labels=np.arange(len(LABELS)), target_names=LABELS,
    zero_division=0, digits=3, output_dict=True,
)
print(classification_report(
    y_validation, y_pred, labels=np.arange(len(LABELS)), target_names=LABELS,
    zero_division=0, digits=3,
))
fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay.from_predictions(
    y_validation, y_pred, labels=np.arange(len(LABELS)), display_labels=LABELS,
    normalize="true", xticks_rotation=45, cmap="Blues", ax=ax,
)
ax.set_title("Untouched validation confusion matrix (row normalized)")
plt.tight_layout()
plt.show()


## 8. Save and log the reproducible bundle

The bundle includes the test-selected Transformer, training-only normalization,
stratified episode assignments, class and feature order, window configuration, history,
and validation report. The same files are attached to the active MLflow run.

In [ ]:
torch.save({
    "model_state_dict": model.state_dict(), "model_config": model_config,
    "feature_columns": list(MODEL_FEATURE_COLUMNS), "labels": LABELS,
    "window_samples": WINDOW_SAMPLES, "sample_dt_s": SAMPLE_DT_S,
    "selected_epoch": checkpoint["epoch"], "selection_test_loss": checkpoint["test_loss"],
}, OUTPUT_DIR / "model.pt")
(OUTPUT_DIR / "preprocessing.json").write_text(json.dumps({
    "feature_columns": list(MODEL_FEATURE_COLUMNS),
    "mean": scaler.mean_.tolist(), "scale": scaler.scale_.tolist(),
}, indent=2))
(OUTPUT_DIR / "split.json").write_text(json.dumps({
    "seed": SEED, "fractions": SPLIT_FRACTIONS, "stratification": "demonstrated_skill_set",
    "episode_ids": split_ids, "window_s": WINDOW_S, "stride_s": STRIDE_S,
    "keep_mixed_windows": KEEP_MIXED_WINDOWS,
}, indent=2))
history.to_csv(OUTPUT_DIR / "history.csv", index=False)
(OUTPUT_DIR / "validation_report.json").write_text(json.dumps(report, indent=2))
def write_window_metadata(split_name, values):
    """Write metadata incrementally without constructing a multi-million-row DataFrame."""
    import pyarrow.parquet as pq

    destination = OUTPUT_DIR / f"{split_name}_windows.parquet"
    writer = None
    episode_dictionary = pa.array(episode_ids)
    label_dictionary = pa.array(LABELS)
    try:
        for left in range(0, len(values["y"]), PREPROCESS_CHUNK_WINDOWS):
            right = min(left + PREPROCESS_CHUNK_WINDOWS, len(values["y"]))
            indices = values["episode_index"][left:right]
            table = pa.table({
                "episode_id": episode_dictionary.take(pa.array(indices, type=pa.int32())),
                "start_time_s": values["start_time_s"][left:right],
                "end_time_s": values["end_time_s"][left:right],
                "mixed_skill": values["mixed_skill"][left:right],
                "label": label_dictionary.take(pa.array(values["y"][left:right], type=pa.int64())),
            })
            writer = writer or pq.ParquetWriter(destination, table.schema, compression="zstd")
            writer.write_table(table)
    finally:
        if writer is not None:
            writer.close()


for split_name, values in raw.items():
    write_window_metadata(split_name, values)

# Do not upload the potentially enormous temporary memory maps as MLflow artifacts.
bundle_files = ["model.pt", "preprocessing.json", "split.json", "history.csv",
                "validation_report.json", *[f"{name}_windows.parquet" for name in raw]]
with mlflow.start_run(run_id=run_id):
    mlflow.log_metric("validation_accuracy", validation_accuracy)
    for filename in bundle_files:
        mlflow.log_artifact(str(OUTPUT_DIR / filename), artifact_path="training_bundle")
print("Saved training bundle to", OUTPUT_DIR.resolve())

if not KEEP_WINDOW_CACHE:
    # Drop every tensor/memmap view before removing backing files (also works on Windows).
    loaders.clear()
    del loaders, dataset, values, raw
    gc.collect()
    shutil.rmtree(CACHE_DIR)
    print("Removed temporary window cache", CACHE_DIR.resolve())


## Interpretation notes

* Results are episode-held-out, not independent random-window performance.
* The test partition is intentionally a **development selection set** in this protocol;
  report the untouched validation metrics as the final generalization estimate.
* Per-class metrics matter because aggregate accuracy can hide weak rare-skill behavior.
* Mixed windows are better suited to a future transition or multi-label model.
* Synthetic performance does not establish real-world tactical generalization.
## Acceleration estimate and measurement protocol

The throughput-first defaults target the actual epoch loop rather than only preprocessing. The
**1.3–2.7× GPU estimate (about 1.9× midpoint)** is intentionally a range because batch scaling,
compiler fusion, CPU input prefetch, asynchronous CUDA transfer, CUDA-resident gathering,
and checkpoint recomputation overlap. The resident loader's **1.05–1.4×** incremental estimate
should be checked by toggling `BVR_CUDA_RESIDENT_DATASET`; it is near 1.0× for a compute-bound
run and can approach 2× only when the input pipeline dominated. These effects overlap. It excludes one-time cache
construction and `torch.compile` warm-up. For a defensible machine-specific number, run once with
the baseline environment printed after training, once with the defaults, discard epoch 1, and
divide the median `train_samples_per_second` values. Keep seed, split, model, and epoch count fixed;
also compare loss curves because changing the physical batch can change optimization behavior even
when the effective batch is reported. Compare `test_seconds` separately as well: the evaluation-batch
optimization affects wall time but not training samples/s. The 1.1–2.0× evaluation estimate and the
worked 1.81× whole-epoch estimate are engineering estimates, not measurements from the target GPU.
